# Gemma 4 E2B — staged training notebook

Trains Gemma 4 E2B through gated stages **S0 → S4**, plotting progress inline and pushing every completed stage to [`kaaninel/we2b`](https://huggingface.co/kaaninel/we2b) so any disconnect can be recovered.

| Stage | What | Gate |
|-------|------|------|
| S0 | Load Gemma 4 E2B + surgery + sanity | ≥1 layer replaced, finite forward |
| S1.distill | Distill warm-start on synthetic | KL halved |
| S1.ws | Workspace-conditional training (held_out) | bank-acc > no-mount |
| S1.route | Router head | top1 ≥ 0.6 |
| S1.G6 | Synthetic G6 (set_W vs set_M) | mount/weights ≥ 0.6 |
| S2 | Real-text distill on FineWeb-Edu | PPL ≤ 1.3× teacher |
| S3 | RAG workspace training on SQuAD | bank-acc ≥ 1.5× no-mount |
| S4 | Final eval bundle + model card | info-only |

Failed gate stops the chain. Auto-resumes from last passed stage on re-run.


## 1. Setup


In [ ]:
# Clone + install (Colab)
import os, subprocess, sys
if not os.path.exists('/content/localsparse'):
    subprocess.run(['git','clone','https://github.com/kaaninel/localsparse.git','/content/localsparse'], check=True)
%cd /content/localsparse
!git pull --quiet
!pip install --quiet -r requirements.txt 2>/dev/null || true
!pip install --quiet 'transformers @ git+https://github.com/huggingface/transformers.git@main' datasets huggingface_hub matplotlib
sys.path.insert(0, '/content/localsparse')


In [ ]:
# HF token (Colab Secrets: HF_TOKEN with write scope)
import os
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        from getpass import getpass
        os.environ['HF_TOKEN'] = getpass('HF_TOKEN (write scope): ')
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('hf login ok')


## 2. Configuration (edit me)


In [ ]:
import torch

# --- Repo + model ---
HF_REPO    = 'kaaninel/we2b'
MODEL_ID   = 'google/gemma-4-E2B'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE      = torch.bfloat16
RUN_ROOT   = '/content/runs/we2b'

# --- Auto-resume? Set False to force restart from S0 ---
AUTO_RESUME = True

# --- Per-stage budgets ---
# S1 synthetic — fast iteration
S1_DISTILL_STEPS   = 800
S1_WS_STEPS        = 1500
S1_ROUTE_STEPS     = 600
S1_G6_MAX_STEPS    = 2000
S1_G6_N_FACTS      = 256

# S2 real-text distill
S2_DISTILL_TOKENS  = 50_000_000   # ~50M tokens. Bump to 100M+ for production.
S2_BATCH_SIZE      = 2
S2_SEQ_LEN         = 1024
S2_GRAD_ACCUM      = 4

# S3 RAG SQuAD
S3_BATCH_SIZE      = 2
S3_MAX_STEPS       = 4000
S3_QA_MAX_LENGTH   = 256
S3_BANK_MAX_LENGTH = 1024

# --- Gates ---
GATE_S1_KL_RATIO       = 0.5
GATE_S1_G6_RATIO       = 0.6
GATE_S2_PPL_MULTIPLIER = 1.3
GATE_S3_RAG_RATIO      = 1.5

print(f'device={DEVICE} dtype={DTYPE}')
print(f'run_root={RUN_ROOT}')
print(f'hf_repo={HF_REPO}')


## 3. Resume check


In [ ]:
import os
from pathlib import Path
Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)

STAGES_IN_ORDER = ['s0_surgery','s1_distill','s1_ws','s1_route','s1_g6',
                   's2_real_distill','s3_rag','s4_final']

from localsparse.hub.checkpointing import HubCheckpointer, install_shutdown_hooks
CKPT = HubCheckpointer(HF_REPO, local_root=RUN_ROOT, token=os.environ['HF_TOKEN'], private=False)

if AUTO_RESUME:
    last = CKPT.latest_completed_stage()
    next_stage = CKPT.resume_or_start(STAGES_IN_ORDER)
    print(f'last completed: {last}')
    print(f'next to run:    {next_stage}')
else:
    next_stage = STAGES_IN_ORDER[0]
    print('AUTO_RESUME=False; starting from', next_stage)

def should_run(stage_id):
    if not AUTO_RESUME:
        return True
    return STAGES_IN_ORDER.index(stage_id) >= STAGES_IN_ORDER.index(next_stage)


## 4. Stage S0 — load Gemma 4 + surgery + sanity

Loads `google/gemma-4-E2B`, runs `surgery_gemma4`, then a single forward to confirm finite logits.


In [ ]:
import json, time, torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer
from localsparse.model.gemma4_adapter import surgery_gemma4, resolve_vocab_size
from localsparse.training.stage_runner import run_stage, gate_always_pass

model = None; tokenizer = None; surgery_report = None

if should_run('s0_surgery'):
    def _s0():
        global model, tokenizer, surgery_report
        tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE)
        surgery_report = surgery_gemma4(model)
        model = model.to(DEVICE)
        global VOCAB_SIZE
        VOCAB_SIZE = resolve_vocab_size(model)
        with torch.no_grad():
            x = torch.randint(0, VOCAB_SIZE, (1, 64), device=DEVICE)
            out = model(input_ids=x, labels=x)
        ok = torch.isfinite(out.logits).all().item()
        return {
            'layers_replaced': len(surgery_report.layers_replaced),
            'layers_skipped': len(surgery_report.layers_skipped),
            'layers_path': surgery_report.layers_path,
            'new_param_bytes': surgery_report.new_param_bytes,
            'inherited_param_bytes': surgery_report.inherited_param_bytes,
            'logits_finite': bool(ok),
            'initial_loss': float(out.loss.detach()),
        }
    def _gate_s0(m):
        ok = (m.get('layers_replaced', 0) >= 1 and m.get('logits_finite'))
        return ok, f"replaced={m.get('layers_replaced')} finite={m.get('logits_finite')}"
    res = run_stage('s0_surgery', _s0, _gate_s0,
                    checkpointer=CKPT, model=model, tokenizer=tokenizer)
    assert res.status == 'pass', f'S0 failed: {res.gate_message}'
else:
    print('skipping s0 — auto-resume past this stage')
    # Re-hydrate model+tokenizer from hub stage
    print('NOTE: re-hydration not yet implemented; falling back to fresh load')
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE)
    surgery_report = surgery_gemma4(model)
    model = model.to(DEVICE)
    VOCAB_SIZE = resolve_vocab_size(model)


In [ ]:
# Install shutdown hooks now that we have a model + checkpointer.
# Best-effort: on Colab disconnect / kernel interrupt, push current state as `-interrupted`.
_CURRENT_STAGE = {'id': 's0_surgery'}
def _interrupt_push():
    if model is None: return
    sid = _CURRENT_STAGE['id']
    from localsparse.hub.checkpointing import StageRecord
    rec = StageRecord(stage_id=sid, status='interrupted', metrics={'note':'shutdown hook'})
    try:
        CKPT.push_stage(stage_id=f'{sid}-interrupted', model=model, tokenizer=tokenizer, record=rec)
    except Exception as e:
        print(f'[interrupt-push] failed: {e}')
install_shutdown_hooks(_interrupt_push)
print('shutdown hooks installed')


## 5. Stage S1 — synthetic mechanism validation

Distill → workspace-conditional → router → G6 eval. Each sub-stage gated.


In [ ]:
# Helpers for plotting train curves
import matplotlib.pyplot as plt
def plot_curve(values, title, ylabel='value'):
    fig, ax = plt.subplots(figsize=(8,3))
    ax.plot(values)
    ax.set_title(title); ax.set_xlabel('step'); ax.set_ylabel(ylabel)
    ax.grid(True, alpha=0.3)
    plt.show()


In [ ]:
# S1.distill — distill post-surgery student against fresh pre-surgery teacher
from localsparse.training.distill import (
    DistillRecipe, distill_warmstart, make_teacher_clone)
from localsparse.training.factoid_world import (
    build_world, render_corpus, make_lm_batches)
from localsparse.training.stage_runner import run_stage, gate_threshold

def _teacher_factory():
    # Reload base Gemma 4 (NO surgery) and freeze.
    from transformers import AutoModelForCausalLM
    print('  loading teacher (fresh, pre-surgery)...')
    t = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE)
    return t.to(DEVICE)

if should_run('s1_distill'):
    _CURRENT_STAGE['id'] = 's1_distill'
    def _s1d():
        teacher = make_teacher_clone(_teacher_factory)
        # Synthetic factoid batches — bridges student to teacher distribution
        world = build_world(vocab_size=VOCAB_SIZE,
                            n_facts=128, seed=7)
        token_stream = render_corpus(world, repeats_per_fact=8)
        batches = make_lm_batches(token_stream, batch_size=2,
                                  seq_len=256, device=DEVICE)
        recipe = DistillRecipe(lr=3e-4, max_steps=S1_DISTILL_STEPS,
                               warmup_steps=100, kl_temperature=2.0,
                               ce_weight=0.1)
        stats = distill_warmstart(student=model, teacher=teacher,
                                  batches=batches, recipe=recipe)
        del teacher
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        if 'loss_history' in stats:
            plot_curve(stats['loss_history'], 'S1.distill loss', 'kl+ce')
        # Derive kl_ratio if present
        out = {k: v for k, v in stats.items()
               if isinstance(v, (int, float, str))}
        if 'initial_kl' in stats and 'final_kl' in stats and stats['initial_kl']:
            out['kl_ratio'] = stats['final_kl'] / stats['initial_kl']
        return out
    res = run_stage('s1_distill', _s1d,
                    gate_threshold('kl_ratio', '<=', GATE_S1_KL_RATIO),
                    checkpointer=CKPT, model=model, tokenizer=tokenizer)
    assert res.status == 'pass', f'S1.distill failed: {res.gate_message}'
else:
    print('skipping s1_distill')


In [ ]:
# S1.ws (workspace-conditional, held_out — bank must matter)
from localsparse.training.workspace_train import (
    WorkspaceTrainRecipe, train_workspace_conditional)
if should_run('s1_ws'):
    _CURRENT_STAGE['id'] = 's1_ws'
    def _s1w():
        recipe = WorkspaceTrainRecipe(
            lr=2e-4, max_steps=S1_WS_STEPS, warmup_steps=100,
            n_facts_per_world=64, qa_per_batch=8, bank_max_length=512,
            n_train_worlds=32,
        )
        stats = train_workspace_conditional(
            model, tokenizer, device=DEVICE, recipe=recipe,
            vocab_size=VOCAB_SIZE, mode='held_out',
            eval_n_facts=64, label_prefix='s1_ws',
        )
        out = {k: v for k, v in stats.items()
               if isinstance(v, (int, float, str))}
        kv = out.get('kv_accuracy') or out.get('kv_acc') or 0
        nm = out.get('no_mount_accuracy') or out.get('nomount_acc') or 0
        out['kv_accuracy'] = float(kv)
        out['no_mount_accuracy'] = float(nm)
        out['kv_minus_nomount'] = float(kv) - float(nm)
        if 'loss_history' in stats:
            plot_curve(stats['loss_history'], 'S1.ws train loss', 'loss')
        return out
    def _gate(m):
        kv = m.get('kv_accuracy',0); nm = m.get('no_mount_accuracy',0)
        ok = kv > nm + 0.05
        return ok, f'kv={kv:.3f} vs no_mount={nm:.3f}'
    res = run_stage('s1_ws', _s1w, _gate,
                    checkpointer=CKPT, model=model, tokenizer=tokenizer)
    assert res.status == 'pass', f'S1.ws failed: {res.gate_message}'
else:
    print('skipping s1_ws')


In [ ]:
# S1.route
from localsparse.training.routing_supervised import RoutingRecipe, train_router
if should_run('s1_route'):
    _CURRENT_STAGE['id'] = 's1_route'
    def _s1r():
        recipe = RoutingRecipe(lr=1e-3, max_steps=S1_ROUTE_STEPS,
                               warmup_steps=50, hidden_size=128,
                               qa_per_batch=16)
        stats = train_router(
            model, device=DEVICE, vocab_size=VOCAB_SIZE,
            n_banks=4, facts_per_bank=32, recipe=recipe,
        )
        out = {k: v for k, v in stats.items()
               if isinstance(v, (int, float, str))}
        if 'loss_history' in stats:
            plot_curve(stats['loss_history'], 'S1.route loss', 'loss')
        return out
    res = run_stage('s1_route', _s1r,
                    gate_threshold('top1', '>=', 0.5),
                    checkpointer=CKPT, model=model, tokenizer=tokenizer)
    assert res.status == 'pass', f'S1.route failed: {res.gate_message}'
else:
    print('skipping s1_route')


In [ ]:
# S1.G6 — synthetic G6 eval (set_W weights-path vs set_M bank-mount)
from localsparse.eval.gemma4_eval import eval_g6_synthetic
from localsparse.training.factoid_world import (
    build_world, render_corpus, make_lm_batches)
from localsparse.training.m15_runners import train_to_convergence

if should_run('s1_g6'):
    _CURRENT_STAGE['id'] = 's1_g6'
    def _s1g():
        vocab = VOCAB_SIZE
        world_W = build_world(vocab_size=vocab, n_facts=S1_G6_N_FACTS, seed=100)
        token_stream = render_corpus(world_W, repeats_per_fact=10)
        batches = make_lm_batches(token_stream, batch_size=4, seq_len=512, device=DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=2e-4)
        stats_W = train_to_convergence(model, batches, opt, max_steps=S1_G6_MAX_STEPS)
        if 'loss_history' in stats_W:
            plot_curve(stats_W['loss_history'], 'S1.G6 weights-path training', 'loss')
        # Now evaluate G6 (weights vs mount)
        g6 = eval_g6_synthetic(
            model, tokenizer, device=DEVICE,
            n_facts=S1_G6_N_FACTS, bank_max_length=1024,
        )
        g6 = {k: v for k, v in g6.items()
              if isinstance(v, (int, float, str))}
        g6['weights_train_loss'] = stats_W.get('final_loss')
        g6['weights_train_steps'] = stats_W.get('steps')
        return g6
    res = run_stage('s1_g6', _s1g,
                    gate_threshold('g6_ratio_mount_over_weights', '>=', GATE_S1_G6_RATIO),
                    checkpointer=CKPT, model=model, tokenizer=tokenizer)
    assert res.status == 'pass', f'S1.G6 failed: {res.gate_message}'
else:
    print('skipping s1_g6')


## 6. Stage S2 — real-text distill on FineWeb-Edu

Preserves Gemma 4 fluency on real English text. Without this, S1's synthetic-only training can drift the model's general-text PPL.


In [ ]:
# S2 — distill on real text, measure PPL gap
from localsparse.training.real_text import streaming_fineweb_batches
from localsparse.training.distill import make_teacher_clone
from localsparse.eval.gemma4_eval import eval_perplexity

if should_run('s2_real_distill'):
    _CURRENT_STAGE['id'] = 's2_real_distill'
    def _s2():
        teacher = make_teacher_clone(_teacher_factory).to(DEVICE).eval()
        ppl_student_pre = eval_perplexity(model, streaming_fineweb_batches(
            tokenizer, batch_size=S2_BATCH_SIZE, seq_len=S2_SEQ_LEN,
            device=DEVICE, seed=42), max_batches=10)
        ppl_teacher = eval_perplexity(teacher, streaming_fineweb_batches(
            tokenizer, batch_size=S2_BATCH_SIZE, seq_len=S2_SEQ_LEN,
            device=DEVICE, seed=42), max_batches=10)
        print(f'  teacher PPL={ppl_teacher["ppl"]:.2f}  student PPL={ppl_student_pre["ppl"]:.2f}')
        opt = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        train_iter = streaming_fineweb_batches(
            tokenizer, batch_size=S2_BATCH_SIZE, seq_len=S2_SEQ_LEN,
            device=DEVICE, n_total_tokens=S2_DISTILL_TOKENS, seed=0,
        )
        kl = torch.nn.KLDivLoss(reduction='batchmean', log_target=False)
        loss_hist = []
        step = 0; accum = 0
        opt.zero_grad()
        for x, y in train_iter:
            student_out = model(input_ids=x, labels=y)
            with torch.no_grad():
                teacher_logits = teacher(input_ids=x).logits
            T = 2.0
            s_lp = torch.log_softmax(student_out.logits / T, dim=-1)
            t_p  = torch.softmax(teacher_logits / T, dim=-1)
            loss = kl(s_lp, t_p) * (T*T) + 0.1 * student_out.loss
            (loss / S2_GRAD_ACCUM).backward()
            accum += 1
            if accum >= S2_GRAD_ACCUM:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step(); opt.zero_grad(); accum = 0
                step += 1
                loss_hist.append(float(loss.detach()))
                if step % 50 == 0:
                    print(f'  s2 step {step}: loss={loss_hist[-1]:.4f}')
        ppl_student_post = eval_perplexity(model, streaming_fineweb_batches(
            tokenizer, batch_size=S2_BATCH_SIZE, seq_len=S2_SEQ_LEN,
            device=DEVICE, seed=42), max_batches=10)
        if loss_hist:
            plot_curve(loss_hist, 'S2 distill loss', 'kl+ce')
        del teacher
        if DEVICE.type == 'cuda': torch.cuda.empty_cache()
        return {
            'ppl_teacher': ppl_teacher['ppl'],
            'ppl_student_pre': ppl_student_pre['ppl'],
            'ppl_student_post': ppl_student_post['ppl'],
            'ppl_ratio': ppl_student_post['ppl'] / ppl_teacher['ppl'],
            'steps': step,
            'tokens_seen': S2_DISTILL_TOKENS,
        }
    def _gate_s2(m):
        r = m.get('ppl_ratio', 999)
        return r <= GATE_S2_PPL_MULTIPLIER, f'ppl_ratio={r:.3f}'
    res = run_stage('s2_real_distill', _s2, _gate_s2,
                    checkpointer=CKPT, model=model, tokenizer=tokenizer)
    assert res.status == 'pass', f'S2 failed: {res.gate_message}'
else:
    print('skipping s2_real_distill')


## 7. Stage S3 — RAG workspace training on SQuAD

Trains the model to answer SQuAD questions by reading the passage from a workspace bank, not from its weights.


In [ ]:
# S3 — RAG SQuAD
from localsparse.training.real_text import rag_batches_from_squad
from localsparse.eval.gemma4_eval import eval_rag_accuracy
from localsparse.workspace.kv_bank import WorkspaceKVBank

if should_run('s3_rag'):
    _CURRENT_STAGE['id'] = 's3_rag'
    def _s3():
        opt = torch.optim.AdamW(model.parameters(), lr=1e-5, weight_decay=0.01)
        train_iter = rag_batches_from_squad(
            tokenizer, batch_size=S3_BATCH_SIZE,
            qa_max_length=S3_QA_MAX_LENGTH, device=DEVICE, seed=0,
        )
        loss_hist = []
        for step in range(S3_MAX_STEPS):
            rb = next(train_iter)
            # one bank per batch — encode the passage
            bank_text = '\n\n'.join(rb.bank_texts)
            bank = WorkspaceKVBank()
            try:
                bank.encode(model, bank_text, tokenizer, DEVICE,
                            max_length=S3_BANK_MAX_LENGTH)
            except Exception as e:
                print(f'  s3 bank encode failed at step {step}: {e}')
                continue
            with bank.inject(model):
                out = model(input_ids=rb.input_ids, labels=rb.labels)
            loss = out.loss
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            loss_hist.append(float(loss.detach()))
            if step % 100 == 0:
                print(f'  s3 step {step}: loss={loss_hist[-1]:.4f}')
        if loss_hist:
            plot_curve(loss_hist, 'S3 RAG loss', 'ce')
        # Eval
        eval_iter = rag_batches_from_squad(
            tokenizer, batch_size=S3_BATCH_SIZE,
            qa_max_length=S3_QA_MAX_LENGTH, device=DEVICE, seed=99,
            split='validation',
        )
        rag = eval_rag_accuracy(model, tokenizer, eval_iter, device=DEVICE,
                                max_batches=16,
                                bank_max_length=S3_BANK_MAX_LENGTH)
        rag['steps'] = S3_MAX_STEPS
        rag['final_loss'] = loss_hist[-1] if loss_hist else None
        return rag
    def _gate_s3(m):
        r = m.get('rag_ratio_mount_over_nomount', 0)
        return r >= GATE_S3_RAG_RATIO, f'rag_ratio={r:.3f}'
    res = run_stage('s3_rag', _s3, _gate_s3,
                    checkpointer=CKPT, model=model, tokenizer=tokenizer)
    assert res.status == 'pass', f'S3 failed: {res.gate_message}'
else:
    print('skipping s3_rag')


## 8. Stage S4 — final eval bundle + model card


In [ ]:
# S4 — final eval bundle
from localsparse.eval.gemma4_eval import eval_g6_synthetic, eval_perplexity, eval_rag_accuracy
from localsparse.training.real_text import streaming_fineweb_batches, rag_batches_from_squad
from localsparse.training.stage_runner import run_stage, gate_always_pass

if should_run('s4_final'):
    _CURRENT_STAGE['id'] = 's4_final'
    def _s4():
        ppl = eval_perplexity(model, streaming_fineweb_batches(
            tokenizer, batch_size=S2_BATCH_SIZE, seq_len=S2_SEQ_LEN,
            device=DEVICE, seed=999), max_batches=20)
        g6 = eval_g6_synthetic(model, tokenizer, device=DEVICE,
                               n_facts=128, bank_max_length=1024)
        rag = eval_rag_accuracy(model, tokenizer,
                                rag_batches_from_squad(tokenizer,
                                    batch_size=S3_BATCH_SIZE,
                                    qa_max_length=S3_QA_MAX_LENGTH,
                                    device=DEVICE, seed=999, split='validation'),
                                device=DEVICE, max_batches=20,
                                bank_max_length=S3_BANK_MAX_LENGTH)
        # Write model card README
        card = ['# we2b — Gemma 4 E2B with workspace KV-bank\n',
                f'Base model: `{MODEL_ID}`\n\n',
                f'## Eval results\n',
                f'- Held-out PPL (FineWeb-Edu): **{ppl["ppl"]:.2f}**\n',
                f'- Synthetic G6 mount/weights ratio: **{g6["g6_ratio_mount_over_weights"]:.3f}**\n',
                f'- SQuAD RAG mount/nomount ratio: **{rag["rag_ratio_mount_over_nomount"]:.3f}**\n']
        readme = ''.join(card)
        readme_path = Path(RUN_ROOT) / 's4_final' / 'README.md'
        readme_path.parent.mkdir(parents=True, exist_ok=True)
        readme_path.write_text(readme)
        # Also push to repo root
        try:
            CKPT.api.upload_file(
                path_or_fileobj=str(readme_path),
                path_in_repo='README.md', repo_id=HF_REPO,
                token=os.environ['HF_TOKEN'],
                commit_message='[s4] update README')
        except Exception as e:
            print('[s4] root readme upload failed:', e)
        return {**ppl, **g6, **rag}
    res = run_stage('s4_final', _s4, gate_always_pass('final eval info-only'),
                    checkpointer=CKPT, model=model, tokenizer=tokenizer)
    print(res.metrics)
else:
    print('skipping s4_final')


## 9. Manual push (run anytime)


In [ ]:
# Manual checkpoint trigger — push current model under a custom tag.
from localsparse.hub.checkpointing import StageRecord
def manual_push(tag='manual'):
    rec = StageRecord(stage_id=tag, status='pass',
                      metrics={'note':'manual trigger'},
                      gate_message='manual push')
    CKPT.push_stage(stage_id=tag, model=model, tokenizer=tokenizer, record=rec)
    print(f'pushed {HF_REPO}/{tag}')
# Example: manual_push('mid-s3-checkpoint')
